# 04 — Imports, Introspection, and Plugin Architecture

Build extensible software using explicit contracts and trusted registries. Learn import-time effects and why dynamic execution of user input is risky.

*Advanced · 90–150 min*

## Objectives
- Inspect functions with inspect.signature
- Understand sys.modules and import caching
- Use explicit allowlisted registries
- Separate plugin interfaces from potentially unsafe arbitrary code loading

## Examples

### 1. Introspection as a diagnostic tool

inspect exposes signatures and annotations for functions and classes. It is helpful when writing adapters or documentation generators, but signatures alone don't express every domain rule.

In [ ]:
import inspect

def compute_total(price: float, rate: float = 0.1) -> float:
    return price * (1 + rate)

print(inspect.signature(compute_total))
print(list(inspect.signature(compute_total).parameters))
print(compute_total.__annotations__)

**Design question:** What validation must a price calculation enforce beyond type annotations?

### 2. Trusted registries instead of eval

A dictionary maps explicit, trusted identifiers to callables. A user chooses from approved names, not arbitrary code strings. Reject unknown names to keep behavior auditable.

In [ ]:
def uppercase(text):
    return text.upper()
def reverse(text):
    return text[::-1]

TRANSFORMS = {"upper": uppercase, "reverse": reverse}
def transform(name, value):
    try:
        operation = TRANSFORMS[name]
    except KeyError as error:
        raise ValueError("unsupported transform") from error
    return operation(value)

print(transform("reverse", "Python"))

**Design question:** What is the threat model if a user can choose arbitrary import paths?

### 3. Module caching and side effects

Python stores loaded modules in sys.modules; repeated imports typically return the same module object. Module top-level code may run during first import, so expensive I/O at import time makes applications harder to test.

In [ ]:
import importlib
import sys

first = importlib.import_module("math")
second = importlib.import_module("math")
print(first is second, "math" in sys.modules)
print(first.sqrt(49))

**Design question:** What should happen at import time in a well-behaved library?

### 4. Explicit dependency injection

Inject a collaborator that implements a documented method instead of scanning globals() for a matching name. A test can substitute a fake without monkey-patching global state.

In [ ]:
class Renderer:
    def render(self, text):
        return f"[{text}]"

def publish(renderer, message):
    return renderer.render(message)

print(publish(Renderer(), "ready"))
print(inspect.signature(Renderer.render))

**Design question:** Why is explicit injection often easier to maintain than plugin discovery at runtime?

## Exercises

**Exercise 1:** Create a plugin registry with a duplicate-key policy.

In [ ]:
# Exercise 1
# TODO: Implement, demonstrate, and add assertions.

**Exercise 2:** Inspect callables to generate a short text signature report.

In [ ]:
# Exercise 2
# TODO: Implement, demonstrate, and add assertions.

**Exercise 3:** Describe import caching and demonstrate object identity for repeated imports.

In [ ]:
# Exercise 3
# TODO: Implement, demonstrate, and add assertions.

**Exercise 4:** Replace a hypothetical eval-based action dispatcher with an allowlisted registry.

In [ ]:
# Exercise 4
# TODO: Implement, demonstrate, and add assertions.

## Common mistakes

- Never evaluate user-supplied Python code to implement plugins.
- Imports may execute module-level side effects.
- Dynamic discovery should be restricted to trusted code and explicit permissions.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** A second `import json` in the same process…

   - **a)** Fails
   - **b)** Reuses the cached module from sys.modules
   - **c)** Re-executes the module

**2.** inspect.signature(f) gives you…

   - **a)** The parameters, defaults and annotations
   - **b)** The call count
   - **c)** The source code

**3.** A safe plugin system runs…

   - **a)** Any module name a user types
   - **b)** Only names on an explicit allowlist
   - **c)** Code from the internet

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/04', {1: 'af6c976d520e', 2: '1a820b8301af', 3: 'be6b892975d0'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [05 — asyncio Event Loops, Coroutines, and Awaitables](05_asyncio_event_loop.ipynb)